Komórka 1: importy

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer

Komórka 2: wczytanie zbioru

In [2]:
data = pd.read_parquet(Path.cwd().parent / "data" / "dataset.parquet")

FEATURES = ["mom_1m", "mom_3m", "mom_6m", "mom_12m", "vol_3m", "dist_high_12m", "liquidity_3m"]

df = data.dropna(subset=FEATURES + ["label"]).reset_index()
df["year"] = df["date"].dt.year
print(df.shape)
df["year"].value_counts().sort_index().to_frame("wierszy").T

(8555, 12)


year,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
wierszy,356,381,429,448,460,486,505,518,530,595,624,625,675,685,699,539


Komórka 3: walidacja walk-forward

In [4]:
FIRST_TEST_YEAR = 2016

def walk_forward(make_model, name):
    """Dla każdego roku testowego trenuje model na wcześniejszych danych i ocenia na tym roku."""
    results, preds = [], []
    for year in range(FIRST_TEST_YEAR, df["year"].max() + 1):
        # Przerwa jednego miesiąca: etykieta z grudnia zależy od zwrotu w styczniu,
        # czyli w okresie testowym, więc grudzień poprzedniego roku pomijamy.
        train = df[df["date"] < pd.Timestamp(f"{year - 1}-12-01")]
        test = df[df["year"] == year]

        model = make_model()
        model.fit(train[FEATURES], train["label"])
        proba = model.predict_proba(test[FEATURES])[:, 1]

        results.append({"rok": year, "trening": len(train), "test": len(test),
                        "AUC": roc_auc_score(test["label"], proba)})
        preds.append(test[["date", "ticker", "label", "fwd_ret"]].assign(score=proba))

    results = pd.DataFrame(results).set_index("rok")
    print(f"{name}: średnie AUC = {results['AUC'].mean():.4f}")
    return results, pd.concat(preds)

Komórka 4: regresja logistyczna

In [5]:
def make_logreg():
    return make_pipeline(
        QuantileTransformer(output_distribution="normal", n_quantiles=500),
        LogisticRegression(max_iter=1000),
    )

logreg_results, logreg_preds = walk_forward(make_logreg, "Regresja logistyczna")
logreg_results.round(4)

Regresja logistyczna: średnie AUC = 0.5218


,trening,test,AUC
rok,,,
2016,2033,486,0.5285
2017,2520,505,0.5089
2018,3021,518,0.4864
2019,3542,530,0.5697
2020,4066,595,0.5637
2021,4657,624,0.5229
2022,5279,625,0.4941
2023,5903,675,0.5448
2024,6575,685,0.5520


Komórka 5: które cechy są ważne dla regresji logistycznej

In [6]:
final_logreg = make_logreg().fit(df[FEATURES], df["label"])
coefs = pd.Series(final_logreg[-1].coef_[0], index=FEATURES).sort_values()
coefs.round(3).to_frame("współczynnik")

,współczynnik
liquidity_3m,-0.040
mom_1m,-0.031
mom_3m,-0.005
vol_3m,0.014
mom_6m,0.019
dist_high_12m,0.036
mom_12m,0.065


Komórka 6: LightGBM

In [7]:
from lightgbm import LGBMClassifier

def make_lgbm():
    return LGBMClassifier(
        n_estimators=200,
        learning_rate=0.03,
        num_leaves=15,          # płytkie drzewa - dane giełdowe są bardzo zaszumione
        min_child_samples=100,  # liść musi mieć co najmniej 100 przykładów
        subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )

lgbm_results, lgbm_preds = walk_forward(make_lgbm, "LightGBM")
lgbm_results.round(4)

LightGBM: średnie AUC = 0.5200


,trening,test,AUC
rok,,,
2016,2033,486,0.5414
2017,2520,505,0.4897
2018,3021,518,0.5080
2019,3542,530,0.5624
2020,4066,595,0.5383
2021,4657,624,0.5132
2022,5279,625,0.5151
2023,5903,675,0.5304
2024,6575,685,0.5199


Komórka 7: porównanie modeli

In [8]:
comparison = pd.DataFrame({
    "Regresja logistyczna": logreg_results["AUC"],
    "LightGBM": lgbm_results["AUC"],
})
print(comparison.mean().round(4))
print("Lat, w których LightGBM > regresja:", (comparison["LightGBM"] > comparison["Regresja logistyczna"]).sum(), "/", len(comparison))
comparison.round(4)

Regresja logistyczna    0.5218
LightGBM                0.5200
dtype: float64
Lat, w których LightGBM > regresja: 5 / 11


,Regresja logistyczna,LightGBM
rok,,
2016,0.5285,0.5414
2017,0.5089,0.4897
2018,0.4864,0.5080
2019,0.5697,0.5624
2020,0.5637,0.5383
2021,0.5229,0.5132
2022,0.4941,0.5151
2023,0.5448,0.5304
2024,0.5520,0.5199


Komórka 8: czy to przekłada się na zysk

In [9]:
def top_k_excess(preds, k=10):
    """Średni zwrot k spółek z najwyższą oceną minus średni zwrot całego uniwersum, miesiąc po miesiącu."""
    top = preds.sort_values("score", ascending=False).groupby("date").head(k).groupby("date")["fwd_ret"].mean()
    excess = top - preds.groupby("date")["fwd_ret"].mean()
    return pd.Series({
        "średnia nadwyżka / miesiąc": excess.mean(),
        "w skali roku (przybliżenie)": excess.mean() * 12,
        "miesięcy z nadwyżką > 0": (excess > 0).mean(),
    })

pd.DataFrame({
    "Regresja logistyczna": top_k_excess(logreg_preds),
    "LightGBM": top_k_excess(lgbm_preds),
}).round(4)

,Regresja logistyczna,LightGBM
średnia nadwyżka / miesiąc,0.0087,0.0039
w skali roku (przybliżenie),0.1040,0.0470
miesięcy z nadwyżką > 0,0.5504,0.5426


Komórka 9: test istotności

In [10]:
def excess_series(preds, k=10):
    top = preds.sort_values("score", ascending=False).groupby("date").head(k).groupby("date")["fwd_ret"].mean()
    return top - preds.groupby("date")["fwd_ret"].mean()

for name, p in [("Regresja logistyczna", logreg_preds), ("LightGBM", lgbm_preds)]:
    e = excess_series(p)
    t = e.mean() / e.std() * np.sqrt(len(e))
    print(f"{name}: {len(e)} miesięcy, t = {t:.2f}")

Regresja logistyczna: 129 miesięcy, t = 2.85
LightGBM: 129 miesięcy, t = 1.43
